# Main experiment: substrate vs main + side chains

Same featurizer (ECFP count, radius 2, 1024 bits), same models and same 10 folds for every input; only the input changes:

| Experiment | Input to ECFP |
|---|---|
| Substrate | whole substrate SMILES |
| Main + Side | main-chain and side-chain ECFP concatenated (side all zeros for substrates without side chains) |
| Substrate + Main + Side | all three concatenated |

The split comes from the splitter on `main` as of 22 Nov 2026 (`modules/features/substrate_split.py`, lemmoor's updated version: non-reacting ring C=O, S=O and OH are side chains, and hand-checked substrates are set in `modules/features/manual_ends.py`). Features are generated with `configs/best_feature_generation.yaml` and models trained with `scripts/models_training.py`. All results are read from `results/newsplit/`; the substrate-only results are the same as before, because they do not depend on the split.

In [1]:
import os

import joblib
import numpy as np
import pandas as pd

results_dir = "../results/newsplit/all/"

EXPERIMENTS = {
    "Substrate": (results_dir, "features_rth_0.99_ecfp_substrate"),
    "Main + Side": (results_dir, "features_rth_0.99_ecfp_main_side"),
    "Substrate + Main + Side": (results_dir, "features_rth_0.99_ecfp_substrate_main_side"),
}

MODEL_DISPLAY_NAMES = {
    "bayes": "BRR",
    "knn": "KNN",
    "lasso": "LR",
    "mlp": "MLP",
    "random_forest": "RF",
    "xgboost": "XGB",
}

In [2]:
results = {"experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}

for experiment, (root_dir, dataset) in EXPERIMENTS.items():
    for model, model_name in MODEL_DISPLAY_NAMES.items():
        scores = joblib.load(os.path.join(root_dir, model, dataset, "scores.pickle"))
        results["experiment"].append(experiment)
        results["model"].append(model_name)
        results["smape_mean"].append(np.mean(scores["smape"]))
        results["smape_std"].append(np.std(scores["smape"]))
        results["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
        results["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))

df_results = pd.DataFrame(results)
df_results

,experiment,model,smape_mean,smape_std,pa_mean,pa_std
0,Substrate,BRR,0.609443,0.107758,0.687972,0.051538
1,Substrate,KNN,0.658913,0.112259,0.634068,0.089120
2,Substrate,LR,0.643621,0.080449,0.679068,0.079777
3,Substrate,MLP,0.652965,0.081939,0.672821,0.074619
4,Substrate,RF,0.658686,0.117282,0.650140,0.128908
5,Substrate,XGB,0.646526,0.117062,0.665734,0.136186
6,Main + Side,BRR,0.621146,0.101824,0.688834,0.060955
7,Main + Side,KNN,0.686790,0.103232,0.630117,0.088188
8,Main + Side,LR,0.693748,0.120896,0.658788,0.073683
9,Main + Side,MLP,0.683055,0.191229,0.648695,0.075108


In [3]:
def to_matrix(values: str) -> pd.DataFrame:
    matrix = df_results.pivot(index="experiment", columns="model", values=values)
    return matrix.loc[list(EXPERIMENTS), list(MODEL_DISPLAY_NAMES.values())]


smape_mean_matrix = to_matrix("smape_mean")
smape_std_matrix = to_matrix("smape_std")
pa_mean_matrix = to_matrix("pa_mean")
pa_std_matrix = to_matrix("pa_std")

smape_matrix = (smape_mean_matrix * 100).round(1).astype(str) + "(" + smape_std_matrix.round(2).astype(str) + ")"
pa_matrix = (pa_mean_matrix * 100).round(1).astype(str) + "(" + pa_std_matrix.round(2).astype(str) + ")"

def smape_pa_table(smape: pd.DataFrame, pa: pd.DataFrame) -> pd.DataFrame:
    return pd.concat({"SMAPE (lower is better)": smape, "PA (higher is better)": pa}, axis=1).round(3)

In [4]:
smape_pa_table(smape_mean_matrix, pa_mean_matrix)

SMAPE (lower is better)                              \
model                                       BRR    KNN     LR    MLP     RF   
experiment                                                                    
Substrate                                 0.609  0.659  0.644  0.653  0.659   
Main + Side                               0.621  0.687  0.694  0.683  0.670   
Substrate + Main + Side                   0.611  0.682  0.645  0.695  0.648   

                               PA (higher is better)                       \
model                      XGB                   BRR    KNN     LR    MLP   
experiment                                                                  
Substrate                0.647                 0.688  0.634  0.679  0.673   
Main + Side              0.661                 0.689  0.630  0.659  0.649   
Substrate + Main + Side  0.678                 0.683  0.621  0.657  0.674   

                                       
model                       RF    XGB  
experiment                             
Substrate                0.650  0.666  
Main + Side              0.646  0.628  
Substrate + Main + Side  0.642  0.606

## Same experiment with the previously best feature sets

Instead of ECFP alone, the featurizer is the feature set that worked best in the greedy forward selection (`greedy_selection_analysis.ipynb`):

| Feature set | Features |
|---|---|
| RF best | EState + functional groups |
| BRR best | custom descriptor + ECFP + EState + functional groups |

Each feature set is computed for the whole substrate, for the main chain + side chains, and for all three together (`configs/best_feature_generation.yaml`), and both models are trained on every variant (`configs/best_cof_matched.yaml`).

In [5]:
best_dir = "../results/newsplit/all/"

BEST_FEATURE_SETS = {"RF best": "rf_best", "BRR best": "brr_best"}
BEST_EXPERIMENTS = {"Substrate": "substrate", "Main + Side": "main_side", "Substrate + Main + Side": "substrate_main_side"}
BEST_MODELS = {"random_forest": "RF", "bayes": "BRR"}

results_best = {"feature_set": [], "experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}

for feature_set, feature_set_name in BEST_FEATURE_SETS.items():
    for experiment, experiment_name in BEST_EXPERIMENTS.items():
        for model, model_name in BEST_MODELS.items():
            scores = joblib.load(os.path.join(best_dir, model, f"features_rth_0.99_{feature_set_name}_{experiment_name}", "scores.pickle"))
            results_best["feature_set"].append(feature_set)
            results_best["experiment"].append(experiment)
            results_best["model"].append(model_name)
            results_best["smape_mean"].append(np.mean(scores["smape"]))
            results_best["smape_std"].append(np.std(scores["smape"]))
            results_best["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
            results_best["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))

df_results_best = pd.DataFrame(results_best)
df_results_best

,feature_set,experiment,model,smape_mean,smape_std,pa_mean,pa_std
0,RF best,Substrate,RF,0.600064,0.105020,0.688485,0.123328
1,RF best,Substrate,BRR,0.659057,0.062595,0.661212,0.060375
2,RF best,Main + Side,RF,0.633075,0.126167,0.629394,0.139968
3,RF best,Main + Side,BRR,0.642463,0.067860,0.662646,0.061995
4,RF best,Substrate + Main + Side,RF,0.612874,0.159212,0.674895,0.111116
5,RF best,Substrate + Main + Side,BRR,0.648968,0.063252,0.661585,0.068324
6,BRR best,Substrate,RF,0.594628,0.131369,0.663916,0.114507
7,BRR best,Substrate,BRR,0.614371,0.097749,0.701282,0.049793
8,BRR best,Main + Side,RF,0.645501,0.093942,0.661329,0.097320
9,BRR best,Main + Side,BRR,0.616008,0.101525,0.706131,0.063556


In [6]:
def to_best_matrix(values: str) -> pd.DataFrame:
    matrix = df_results_best.pivot(index=["feature_set", "experiment"], columns="model", values=values)
    rows = [(feature_set, experiment) for feature_set in BEST_FEATURE_SETS for experiment in BEST_EXPERIMENTS]
    return matrix.loc[rows, list(BEST_MODELS.values())]


smape_mean_matrix_best = to_best_matrix("smape_mean")
pa_mean_matrix_best = to_best_matrix("pa_mean")

In [7]:
smape_pa_table(smape_mean_matrix_best, pa_mean_matrix_best)

SMAPE (lower is better)         \
model                                                    RF    BRR   
feature_set experiment                                               
RF best     Substrate                                 0.600  0.659   
            Main + Side                               0.633  0.642   
            Substrate + Main + Side                   0.613  0.649   
BRR best    Substrate                                 0.595  0.614   
            Main + Side                               0.646  0.616   
            Substrate + Main + Side                   0.600  0.611   

                                    PA (higher is better)         
model                                                  RF    BRR  
feature_set experiment                                            
RF best     Substrate                               0.688  0.661  
            Main + Side                             0.629  0.663  
            Substrate + Main + Side                 0.675  0.662  
BRR best    Substrate                               0.664  0.701  
            Main + Side                             0.661  0.706  
            Substrate + Main + Side                 0.655  0.693

## Same experiment evaluated on CTFs

The main/side split is most reliable for the 29 CTF substrates (15 of them have side chains). Two protocols:

| Protocol | Training | Test | Script and config |
|---|---|---|---|
| CTFs only | CTFs | CTFs (3 folds x 4 repeats) | `scripts/models_training.py`, `configs/best_ctf_only.yaml` |
| CTF test | half of the CTFs + all other COFs | the other half of the CTFs (2 folds x 5 repeats) | `scripts/models_training_cft_cof.py`, `configs/best_ctf_test.yaml` |

SMAPE of predicting the training median: 0.396 (CTFs only) and 0.441 (CTF test). These numbers are not comparable with the tables above, which are evaluated on all substrates.

In [8]:
ctf_only_dir = "../results/newsplit/ctf_only/"
ctf_test_dir = "../results/newsplit/ctf_test/"

CTF_FEATURE_SETS = {"ECFP": "ecfp", **BEST_FEATURE_SETS}


def load_ctf_results(root_dir: str) -> pd.DataFrame:
    results_ctf = {"feature_set": [], "experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}
    for feature_set, feature_set_name in CTF_FEATURE_SETS.items():
        for experiment, experiment_name in BEST_EXPERIMENTS.items():
            for model, model_name in BEST_MODELS.items():
                scores = joblib.load(os.path.join(root_dir, model, f"features_rth_0.99_{feature_set_name}_{experiment_name}", "scores.pickle"))
                results_ctf["feature_set"].append(feature_set)
                results_ctf["experiment"].append(experiment)
                results_ctf["model"].append(model_name)
                results_ctf["smape_mean"].append(np.mean(scores["smape"]))
                results_ctf["smape_std"].append(np.std(scores["smape"]))
                results_ctf["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
                results_ctf["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))
    return pd.DataFrame(results_ctf)


def to_ctf_matrix(df: pd.DataFrame, values: str) -> pd.DataFrame:
    matrix = df.pivot(index=["feature_set", "experiment"], columns="model", values=values)
    rows = [(feature_set, experiment) for feature_set in CTF_FEATURE_SETS for experiment in BEST_EXPERIMENTS]
    return matrix.loc[rows, list(BEST_MODELS.values())]


df_results_ctf_only = load_ctf_results(ctf_only_dir)
df_results_ctf_test = load_ctf_results(ctf_test_dir)

### CTFs only: SMAPE and PA

In [9]:
smape_pa_table(to_ctf_matrix(df_results_ctf_only, "smape_mean"), to_ctf_matrix(df_results_ctf_only, "pa_mean"))

SMAPE (lower is better)         \
model                                                    RF    BRR   
feature_set experiment                                               
ECFP        Substrate                                 0.375  0.402   
            Main + Side                               0.382  0.411   
            Substrate + Main + Side                   0.392  0.398   
RF best     Substrate                                 0.446  0.416   
            Main + Side                               0.377  0.424   
            Substrate + Main + Side                   0.402  0.420   
BRR best    Substrate                                 0.417  0.406   
            Main + Side                               0.364  0.447   
            Substrate + Main + Side                   0.386  0.426   

                                    PA (higher is better)         
model                                                  RF    BRR  
feature_set experiment                                            
ECFP        Substrate                               0.607  0.564  
            Main + Side                             0.539  0.565  
            Substrate + Main + Side                 0.564  0.558  
RF best     Substrate                               0.471  0.564  
            Main + Side                             0.576  0.553  
            Substrate + Main + Side                 0.540  0.564  
BRR best    Substrate                               0.530  0.572  
            Main + Side                             0.606  0.557  
            Substrate + Main + Side                 0.532  0.574

### CTF test (other COFs in training): SMAPE and PA

In [10]:
smape_pa_table(to_ctf_matrix(df_results_ctf_test, "smape_mean"), to_ctf_matrix(df_results_ctf_test, "pa_mean"))

SMAPE (lower is better)         \
model                                                    RF    BRR   
feature_set experiment                                               
ECFP        Substrate                                 0.418  0.356   
            Main + Side                               0.405  0.364   
            Substrate + Main + Side                   0.416  0.349   
RF best     Substrate                                 0.472  0.361   
            Main + Side                               0.454  0.365   
            Substrate + Main + Side                   0.417  0.363   
BRR best    Substrate                                 0.412  0.350   
            Main + Side                               0.460  0.344   
            Substrate + Main + Side                   0.400  0.345   

                                    PA (higher is better)         
model                                                  RF    BRR  
feature_set experiment                                            
ECFP        Substrate                               0.617  0.633  
            Main + Side                             0.575  0.627  
            Substrate + Main + Side                 0.577  0.638  
RF best     Substrate                               0.508  0.579  
            Main + Side                             0.511  0.623  
            Substrate + Main + Side                 0.533  0.616  
BRR best    Substrate                               0.557  0.643  
            Main + Side                             0.534  0.645  
            Substrate + Main + Side                 0.579  0.644